In [ ]:
class FrequencyChannelAttention(nn.Module):
    def __init__(self, channels, reduction=16):
        super(FrequencyChannelAttention, self).__init__()
        self.channels = channels
        self.reduction = reduction

        # Squeeze operation in frequency domain
        self.squeeze = nn.Conv2d(channels, channels // reduction, kernel_size=1)
        self.excitation = nn.Conv2d(channels // reduction, channels, kernel_size=1)

    def forward(self, x):
        # Apply DCT along spatial dimensions to get frequency representation
        freq = torch.fft.fft2(x, norm="ortho")

        # Take the magnitude of the frequency representation
        freq_magnitude = torch.abs(freq)

        # Squeeze and Excitation in frequency domain
        squeezed = self.squeeze(freq_magnitude)
        activated = torch.relu(squeezed)
        excited = self.excitation(activated)

        # Apply a sigmoid to get channel-wise attention weights
        attention = torch.sigmoid(excited)

        # Apply attention weights to original input
        x = x * attention

        return x